# Explore the COAWST Dataset with Cubed + Lithops on AWS Lambda

This notebook is a variant of `COAWST_explore.ipynb` that:

* Reads data from the **Icechunk** virtual store on S3 (`s3://usgs-coawst/useast-archive/icechunk/coawst-useast.icechunk`)
* Replaces Dask with [Cubed](https://cubed-dev.github.io/cubed/) for chunked array computation
* Uses [Lithops](https://lithops-cloud.github.io/) to execute tasks on **AWS Lambda** instead of a persistent cluster

### Cubed + Lithops vs Dask

| | Dask | Cubed + Lithops |
|---|---|---|
| Scaling | Persistent cluster (workers always running) | Serverless — Lambda functions on demand |
| Cost | Pay for idle workers | Pay only for compute actually used |
| Intermediate storage | In-memory / spill to disk | S3 (cubed `work_dir`) |
| Integration | `dask`-backed xarray arrays | `cubed`-backed xarray arrays via `cubed-xarray` |


## Prerequisites

To run this notebook with Lithops on AWS Lambda you need:

### 1. Local conda environment

```bash
conda env create -f coawst-icechunk-env.yml
conda activate coawst-icechunk
```

Key packages: `cubed`, `cubed-xarray`, `lithops`, `icechunk`, `zarr==3.1.6`  
(`zarr` is pinned to 3.1.6 — cubed 0.26 requires `RegularChunkGrid` which was removed in zarr 3.2)

### 2. AWS account with the following resources

| Resource | Purpose |
|---|---|
| S3 bucket | Lithops job storage and cubed intermediate arrays |
| ECR repository | Stores the Lambda container image (created automatically by `lithops runtime build`) |
| Lambda | Executes cubed tasks |

### 3. IAM role for Lambda workers

Create a role (e.g. `lambdaLithopsExecutionRole`) with a trust policy allowing `lambda.amazonaws.com` to assume it, and attach these policies:

- `AWSLambdaBasicExecutionRole` (CloudWatch logs)
- Inline policy granting `s3:GetObject`, `s3:PutObject`, `s3:DeleteObject`, `s3:ListBucket` on your Lithops S3 bucket

### 4. Build and deploy the Lambda container runtime

The runtime packages all Python dependencies into a Docker image pushed to ECR and deployed as a Lambda function.
Run these commands **once** (and again whenever you change `Dockerfile.lithops`):

```bash
# Build image and push to ECR  (--no-cache avoids stale zarr layers)
AWS_PROFILE=YOUR_PROFILE conda run -n coawst-icechunk \
    lithops runtime build -b aws_lambda -f Dockerfile.lithops YOUR_RUNTIME_NAME --no-cache

# Deploy as a Lambda function
AWS_PROFILE=YOUR_PROFILE conda run -n coawst-icechunk \
    lithops runtime deploy -b aws_lambda YOUR_RUNTIME_NAME
```

To verify the runtime is working:

```bash
AWS_PROFILE=YOUR_PROFILE conda run -n coawst-icechunk python -c "
import lithops
fexec = lithops.FunctionExecutor()
fexec.map(lambda x: f'Hello {x}!', ['Lambda'])
print(fexec.get_result())
"
```

### 5. Lithops config file (`~/.lithops/config`)

See the next cell for the required contents.

### Notes on credentials

- **Local process** uses the named AWS profile (`aws_profile` in config) to submit jobs and read results.
- **Lambda workers** use the attached IAM role — no profile is shipped to Lambda.
- **Icechunk data** at `s3://usgs-coawst` is a public AWS Open Data bucket — Lambda workers access it anonymously.

In [ ]:
import os
import numpy as np
import xarray as xr
import zarr
import icechunk
from icechunk import (
    ManifestConfig,
    ManifestSplitCondition,
    ManifestSplitDimCondition,
    ManifestSplittingConfig,
)
import hvplot.xarray
import cf_xarray
import panel as pn
from matplotlib import path
import cubed
import cubed_xarray  # registers the cubed backend with xarray
from cubed.runtime.executors.lithops import LithopsExecutor

import logging
logging.getLogger("lithops").setLevel(logging.WARNING)

## Configure Lithops and Cubed

### `~/.lithops/config`

Create this file before running the notebook:

```yaml
lithops:
  backend: aws_lambda
  storage: aws_s3

aws:
  region: us-west-2
  aws_profile: YOUR_AWS_PROFILE

aws_lambda:
  execution_role: arn:aws:iam::ACCOUNT_ID:role/lambdaLithopsExecutionRole
  runtime: YOUR_RUNTIME_NAME
  runtime_memory: 2048
  runtime_timeout: 600

aws_s3:
  storage_bucket: YOUR_LITHOPS_BUCKET
```

### Rebuilding the runtime

If you change `Dockerfile.lithops` or update any package versions, delete the old Lambda function and redeploy:

```bash
# Find the function name
AWS_PROFILE=YOUR_PROFILE aws lambda list-functions --region us-west-2 \
    --query 'Functions[?starts_with(FunctionName, `lithops-worker`)].FunctionName'

# Delete it, then rebuild and redeploy
AWS_PROFILE=YOUR_PROFILE aws lambda delete-function --region us-west-2 \
    --function-name FUNCTION_NAME

AWS_PROFILE=YOUR_PROFILE conda run -n coawst-icechunk \
    lithops runtime build -b aws_lambda -f Dockerfile.lithops YOUR_RUNTIME_NAME --no-cache

AWS_PROFILE=YOUR_PROFILE conda run -n coawst-icechunk \
    lithops runtime deploy -b aws_lambda YOUR_RUNTIME_NAME
```

### Debugging Lambda failures

If Lambda workers fail silently (0% completion), check CloudWatch logs:

```bash
AWS_PROFILE=YOUR_PROFILE aws logs describe-log-groups --region us-west-2 \
    --log-group-name-prefix /aws/lambda/lithops-worker

AWS_PROFILE=YOUR_PROFILE aws logs get-log-events --region us-west-2 \
    --log-group-name LOG_GROUP --log-stream-name 'STREAM_NAME'
```

In [ ]:
os.environ['AWS_PROFILE'] = 'esiplab2'  # local process; Lambda workers use IAM role

# Work dir must be accessible by both local process (via AWS_PROFILE) AND Lambda
# workers (via IAM role). The lithops bucket is in the same account as the Lambda
# role, so both can access it without profile-based credentials.
WORK_DIR = 's3://lithops-us-west-2-4o4w/cubed-tmp'

spec = cubed.Spec(
    work_dir=WORK_DIR,
    allowed_mem='500MB',
)

executor = LithopsExecutor(
    config={
        'aws': {'region': 'us-west-2', 'aws_profile': 'esiplab2'},
        'aws_lambda': {
            'execution_role': 'arn:aws:iam::097532040392:role/lambdaLithopsExecutionRole',
            'runtime': 'coawst-icechunk-312',
            'runtime_memory': 2048,
            'runtime_timeout': 600,
        },
        'aws_s3': {'storage_bucket': 'lithops-us-west-2-4o4w'},
    }
)


## Open Dataset

Open the Icechunk store anonymously, then open the dataset with xarray using
`chunked_array_type="cubed"` so that array variables are backed by cubed arrays
rather than dask arrays.


In [ ]:
bucket = 'usgs-coawst'
region = 'us-west-2'
prefix = 'useast-archive/icechunk/coawst-useast.icechunk'
TIME_DIM = 'ocean_time'

# Manifest splitting mirrors the config used when the store was built.
# Icechunk uses it for lazy manifest loading: only the manifest files
# covering the requested time range are fetched, not the entire store.
split_config = ManifestSplittingConfig.from_dict({
    ManifestSplitCondition.AnyArray(): {
        ManifestSplitDimCondition.DimensionName(TIME_DIM): 365 * 24
    }
})
config = icechunk.RepositoryConfig(manifest=ManifestConfig(splitting=split_config))
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f's3://{bucket}/',
        store=icechunk.s3_store(region=region, anonymous=True),
    )
)
creds = icechunk.containers_credentials(
    {f's3://{bucket}/': icechunk.s3_credentials(anonymous=True)}
)
storage = icechunk.s3_storage(bucket=bucket, prefix=prefix, region=region, anonymous=True)
repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=creds)
session = repo.readonly_session('main')


In [ ]:
%%time
ds = xr.open_zarr(
    session.store,
    consolidated=False,
    chunked_array_type='cubed',
    from_array_kwargs={'spec': spec},
)
ds


The dataset repr looks similar to the Dask version, but array variables are now
backed by cubed arrays. Metadata and coordinates are already loaded; data variables
remain lazy.

In [ ]:
print(f"Dataset size (uncompressed): {ds.nbytes / 1e12:.2f} TB")

In [ ]:
var = 'Hwave'
da = ds[var]

x = da.cf['longitude']
y = da.cf['latitude']
t = da.cf['time']
print(f"lon: {x.name}, lat: {y.name}, time: {t.name}")
da

## Example: Load the entire spatial domain at a single time step

This reads only ~8 chunks and is fast even locally — Lithops adds overhead for small
jobs, so we use the local executor here and only invoke Lithops for large computations.

In [ ]:
%%time
# Small job — run locally (no Lithops overhead)
da2d = da.cf.sel(T='2012-10-29 12:00', method='nearest').compute()
da2d

In [ ]:
da2d.hvplot.quadmesh(x=x.name, y=y.name, rasterize=True, geo=True, tiles='OSM', cmap='viridis')

## Example: Load a time series at a specific lon, lat location

Since cubed arrays don't support vectorized indexing, we use scipy's `cKDTree`
directly to find the nearest grid (j, i) indices from the 2D lat/lon arrays,
then select with `isel()` — basic integer indexing that cubed supports.


In [ ]:
lat, lon = 42.5, -70.0  # Gulf of Maine, ~100 km east of Boston

# cubed arrays don't support vectorized indexing, so we use scipy's cKDTree
# to find the nearest (j, i) grid indices, then select with isel().
from scipy.spatial import cKDTree

x_np = x.compute().values
y_np = y.compute().values

tree = cKDTree(np.column_stack([y_np.ravel(), x_np.ravel()]))
_, flat_idx = tree.query([[lat, lon]])
j, i = np.unravel_index(flat_idx[0], y_np.shape)
print(f'Nearest grid point: eta_rho={j}, xi_rho={i}')


In [ ]:
# Check how many chunks are needed for one month
da.isel(eta_rho=j, xi_rho=i).cf.sel(T='2012-10')

In [ ]:
%%time
# ~5 chunks — still fast locally
da1d = da.isel(eta_rho=j, xi_rho=i).cf.sel(T='2012-10').compute()
da1d.hvplot(x=t.name, grid=True)

In [ ]:
# Full record = 669 chunks — worth using Lithops
da.isel(eta_rho=j, xi_rho=i)

For the full time series (669 chunks) we invoke the Lithops executor. Cubed dispatches
each chunk as an independent Lambda function — no persistent cluster needed.

Note: the first call incurs Lambda **cold-start** latency (~5–15 s). Subsequent calls
reuse warm containers and are faster.

In [ ]:
%%time
da_full = da.isel(eta_rho=j, xi_rho=i).compute(executor=executor)
da_full.hvplot(x=t.name, grid=True)


## Example: Compute the time mean over the entire domain for a time period

This requires reading all spatial chunks for every time step in the period — easily
hundreds of chunks. Cubed decomposes this into blockwise tasks and Lithops runs them
in parallel across many Lambda invocations.

In [ ]:
da_year = da.cf.sel(T=slice('2016-01-01', '2017-01-01'))
da_mean_lazy = da_year.mean(dim=t.name)


In [ ]:
%%time
da_mean = da_mean_lazy.compute(executor=executor)
da_mean.hvplot.quadmesh(x=x.name, y=y.name, rasterize=True, geo=True, tiles='OSM', cmap='viridis')


## Example: Subset a region and time range, export to NetCDF

In [ ]:
def bbox2ij(lon, lat, bbox=[-160., -155., 18., 23.]):
    """Return i,j index ranges that fully cover a lat/lon bounding box."""
    bbox = np.array(bbox)
    mypath = np.array([bbox[[0, 1, 1, 0]], bbox[[2, 2, 3, 3]]]).T
    p = path.Path(mypath)
    points = np.vstack((lon.ravel(), lat.ravel())).T
    n, m = np.shape(lon)
    inside = p.contains_points(points).reshape((n, m))
    ii, jj = np.meshgrid(range(m), range(n))
    return min(ii[inside]), max(ii[inside]), min(jj[inside]), max(jj[inside])

In [ ]:
bbox = [-76.63, -73.56, 37.58, 41.23]  # Delaware River Basin

# x_np and y_np are already computed numpy arrays from cell 14
i0, i1, j0, j1 = bbox2ij(x_np, y_np, bbox=bbox)
print(i0, i1, j0, j1)

ds_drb = ds[['temp', 'salt', 'Hwave']].isel(eta_rho=slice(j0, j1), xi_rho=slice(i0, i1))
ds_drb_week = ds_drb.cf.sel(T=slice('2022-04-01', '2022-04-08'))
print(f'Subset size: {ds_drb_week.nbytes / 1e6:.1f} MB')
ds_drb_week


In [ ]:
%%time
da_drb = ds_drb_week['salt'].compute(executor=executor)

viz = da_drb.hvplot.quadmesh(
    x=x.name, y=y.name, geo=True,
    cmap='turbo', rasterize=True, tiles='OSM', title='salt'
)
pn.panel(viz, widgets={ds_drb_week.cf.coordinates['time'][0]: pn.widgets.Select}).servable('DRB Explorer')


In [ ]:
%%time
encoding = {
    v: dict(zlib=True, complevel=4, fletcher32=False, shuffle=True, _FillValue=None)
    for v in ds_drb_week.variables
}
ds_drb_week.compute(executor=executor).to_netcdf('drb.nc', encoding=encoding, mode='w')
